# Segment Anything: a promptable pretrained segmenter

The chapter-8 lesson in a new modality — a foundation model for segmentation, prompted with a point or a box, with no training at all.

**Runs on:** GPU recommended · downloads ~400 MB of weights &nbsp;·&nbsp; **Slides:** [Chapter 11 — Image Segmentation](../../../course-web-slides/ch11/index.html) &nbsp;·&nbsp; **Section:** 03 — Segment Anything

---

## Loading it

In [ ]:
import keras
import keras_hub
import numpy as np
import matplotlib.pyplot as plt

model = keras_hub.models.SAMImageSegmenter.from_preset("sam_base_sa1b")
print(type(model).__name__)

**SAM was trained on 1.1 billion masks over 11 million images.** Like the backbones in chapter 8, everything it knows was learned before this notebook started.

## An image

In [ ]:
image_path = keras.utils.get_file(
    origin="https://img-datasets.s3.amazonaws.com/elephant.jpg")
image = np.array(keras.utils.load_img(image_path, target_size=(1024, 1024)))

plt.figure(figsize=(6, 6))
plt.imshow(image); plt.axis("off"); plt.show()

## Prompting with a point

In [ ]:
def show_mask(mask, ax, color=(0.13, 0.55, 0.85, 0.6)):
    h, w = mask.shape[-2:]
    ax.imshow(mask.reshape(h, w, 1) * np.array(color).reshape(1, 1, -1))

point = np.array([[[580.0, 450.0]]])          # (batch, num_points, 2)
label = np.array([[1]])                        # 1 = foreground, 0 = background

outputs = model.predict({
    "images": image[np.newaxis, ...].astype("float32"),
    "points": point,
    "labels": label,
}, verbose=0)

mask = outputs["masks"][0][0] > 0.0
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image)
show_mask(mask, ax)
ax.scatter(point[0, :, 0], point[0, :, 1], c="yellow", s=200, marker="*",
           edgecolors="k")
ax.axis("off"); ax.set_title("One point, one mask")
plt.show()

One click, no training, no labels. **This is the chapter-8 argument again**: the largest single jump available is not architecture, it is starting from something already trained.

## The ambiguity SAM handles explicitly

In [ ]:
# A point on a person could mean the shirt, the torso, or the whole person.
# SAM returns several masks and a confidence for each.
all_masks = outputs["masks"][0]
iou = outputs["iou_pred"][0]

fig, axes = plt.subplots(1, len(all_masks), figsize=(4 * len(all_masks), 4.4))
for ax, m, score in zip(np.atleast_1d(axes), all_masks, iou):
    ax.imshow(image); show_mask(m > 0.0, ax)
    ax.set_title(f"predicted IoU {float(score):.2f}", fontsize=10)
    ax.axis("off")
plt.suptitle("A point is ambiguous, so several masks are returned", y=1.02)
plt.tight_layout(); plt.show()

**The ambiguity is in the prompt, not the model**, and SAM does not pretend otherwise. Returning several candidates with confidences is a better design than picking one and being confidently wrong — a pattern worth borrowing.

## Prompting with a box

In [ ]:
box = np.array([[[300.0, 200.0], [900.0, 800.0]]])   # two corners

outputs = model.predict({
    "images": image[np.newaxis, ...].astype("float32"),
    "boxes": box,
}, verbose=0)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image)
show_mask(outputs["masks"][0][0] > 0.0, ax)
x0, y0 = box[0, 0]; x1, y1 = box[0, 1]
ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                           fill=False, edgecolor="yellow", lw=2))
ax.axis("off"); ax.set_title("Box prompt — far less ambiguous")
plt.show()

## When to reach for this, and when not

| Situation | What to do |
|---|---|
| Generic objects, no labelled data | **SAM, zero-shot.** Nothing you train in a week will compete. |
| A specific domain — cells, defects, satellite imagery | SAM **as a labelling accelerator**, then fine-tune a smaller model on the masks it helped you produce. |
| Fixed classes, plenty of labels, tight latency budget | Notebook 01's model, or a U-Net. SAM is large and general, and you are paying for generality you do not need. |

The middle row is the one most projects land on, and it is worth planning for: **the fastest route to a labelled dataset is often a foundation model plus a human correcting it.**

---

## What to take away

- SAM segments generic objects with no training, prompted by a point or a box.
- It returns several masks with confidences, because a point prompt is genuinely ambiguous.
- The chapter-8 lesson holds in a new modality: pretraining beats architecture.
- In a specialised domain, use it to build the labelled dataset, then train something small.